# imports

In [27]:
import mysql.connector
import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Path relative to Scripts/
data_dir = Path("../Data")
input_file = data_dir / "bank_dataset_cleaned_2026-06-29.csv"

df = pd.read_csv(input_file, encoding="utf-8-sig", index_col=0)

print(f"Dataset loaded from: {input_file.resolve()}")
print(f"Shape: {df.shape}")

Dataset loaded from: C:\Users\nowan\Documents\itacademy\Simulador\ProjecteData\Equip_32\Data\bank_dataset_cleaned_2026-07-01.csv
Shape: (9519, 19)


In [34]:
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,no_previous_contact,had_previous_contact
id,,,,,,,,,,,,,,,,,,,
1,59.0,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,1,1,0
4,55.0,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,1,1,0
5,54.0,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,1,1,0
6,42.0,management,single,tertiary,no,0,yes,yes,unknown,5,may,562,2,-1,0,unknown,1,1,0
8,60.0,retired,divorced,secondary,no,545,yes,no,unknown,6,may,1030,1,-1,0,unknown,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10638,33.0,technician,married,secondary,no,218,yes,yes,telephone,2,mar,169,4,-1,0,unknown,0,1,0
10639,42.0,management,single,tertiary,no,1146,yes,no,unknown,15,may,98,2,-1,0,unknown,0,1,0
10640,31.0,unemployed,single,unknown,no,167,no,no,cellular,20,nov,316,1,-1,0,unknown,0,1,0


# Data Transformations

# 1 Demografical clustering

In [35]:
# Age_group
bins   = [17, 25, 35, 50, 65, 100]
labels = [
    "Young (18-25)",
    "Young Adult (26-35)",
    "Adult (36-50)",
    "Middle-Aged (51-65)",
    "Senior (65+)"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels,
    right=True    # right-closed intervals: (17,25] includes 25
)

In [36]:
counts = df["age_group"].value_counts().sort_index()
print(counts)
print(f"\nUnassigned (NaN): {df['age_group'].isna().sum()}")

age_group
Young (18-25)           410
Young Adult (26-35)    3359
Adult (36-50)          3617
Middle-Aged (51-65)    1759
Senior (65+)            374
Name: count, dtype: int64

Unassigned (NaN): 0


In [37]:
age_summary = (
    df.groupby("age_group", observed=True)["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

age_summary["n_clients"] = df.groupby("age_group", observed=True).size()
print(age_summary)

deposit              pct_no  pct_yes  n_clients
age_group                                      
Young (18-25)        0.2707   0.7293        410
Young Adult (26-35)  0.5138   0.4862       3359
Adult (36-50)        0.5809   0.4191       3617
Middle-Aged (51-65)  0.4997   0.5003       1759
Senior (65+)         0.1845   0.8155        374


In [38]:
# Financial Burden

# "unknown" is treated as 0 (no burden assumed)
# This is a deliberate modelling choice — document it in the notebook

burden_map = {"yes": 1, "no": 0, "unknown": 0}

df["default_score"]  = df["default"].map(burden_map)
df["housing_score"]  = df["housing"].map(burden_map)
df["loan_score"]     = df["loan"].map(burden_map)

In [39]:
df["financial_burden"] = (
    df["default_score"] +
    df["housing_score"] +
    df["loan_score"]
)

In [40]:
burden_labels = {
    0: "No burden",
    1: "Low burden",
    2: "Medium burden",
    3: "High burden"
}

df["financial_burden_label"] = df["financial_burden"].map(burden_labels)

In [41]:
burden_summary = (
    df.groupby("financial_burden_label")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

burden_summary["n_clients"] = df.groupby("financial_burden_label").size()

# Sort by score for readability
burden_summary = burden_summary.reindex(burden_labels.values())
print(burden_summary)

deposit                 pct_no  pct_yes  n_clients
financial_burden_label                            
No burden               0.3779   0.6221       4554
Low burden              0.6275   0.3725       4212
Medium burden           0.6944   0.3056        733
High burden             0.6500   0.3500         20


In [42]:
# education

print(df["education"].value_counts())
print(f"\nUnknown count: {(df['education'] == 'unknown').sum()}")


education
secondary    4647
tertiary     3188
primary      1259
unknown       425
Name: count, dtype: int64

Unknown count: 425


In [43]:
# Ordinal scale: unknown → NaN (excluded from ranking)
# primary=1, secondary=2, tertiary=3

education_order = {
    "primary"   : 1,
    "secondary" : 2,
    "tertiary"  : 3,
    "unknown"   : None
}

df["education_rank"] = df["education"].map(education_order)

In [44]:
education_labels = {
    "primary"   : "Primary",
    "secondary" : "Secondary",
    "tertiary"  : "Tertiary",
    "unknown"   : "Unknown"
}

df["education_label"] = df["education"].map(education_labels)

In [45]:
edu_summary = (
    df.groupby("education_label")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

edu_summary["n_clients"] = df.groupby("education_label").size()

# Sort by ordinal rank
order = ["Primary", "Secondary", "Tertiary", "Unknown"]
edu_summary = edu_summary.reindex(order)
print(edu_summary)

deposit          pct_no  pct_yes  n_clients
education_label                            
Primary          0.5965   0.4035       1259
Secondary        0.5425   0.4575       4647
Tertiary         0.4448   0.5552       3188
Unknown          0.4612   0.5388        425


In [46]:
#jobs

print(df["job"].value_counts())
print(f"\nUnknown count: {(df['job'] == 'unknown').sum()}")

job
management       2190
blue-collar      1618
technician       1541
admin.           1151
services          780
retired           699
self-employed     341
student           334
unemployed        307
entrepreneur      273
housemaid         226
unknown            59
Name: count, dtype: int64

Unknown count: 59


In [47]:
job_profile_map = {
    "admin."       : "White Collar",
    "management"   : "White Collar",
    "technician"   : "White Collar",
    "blue-collar"  : "Blue Collar",
    "housemaid"    : "Blue Collar",
    "services"     : "Blue Collar",
    "entrepreneur" : "Self-Employed",
    "self-employed": "Self-Employed",
    "retired"      : "Retired",
    "student"      : "Student",
    "unemployed"   : "Unemployed/Unknown",
    "unknown"      : "Unemployed/Unknown"
}

df["job_profile"] = df["job"].map(job_profile_map)

In [48]:
job_summary = (
    df.groupby("job_profile")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

job_summary["n_clients"] = df.groupby("job_profile").size()

order = ["White Collar", "Blue Collar", "Self-Employed", "Unemployed/Unknown", "Retired", "Student"]
job_summary = job_summary.reindex(order)
print(job_summary)

deposit             pct_no  pct_yes  n_clients
job_profile                                   
White Collar        0.5018   0.4982       4882
Blue Collar         0.6220   0.3780       2624
Self-Employed       0.5749   0.4251        614
Unemployed/Unknown  0.4235   0.5765        366
Retired             0.3119   0.6881        699
Student             0.2335   0.7665        334


In [49]:
df


,age,job,marital,education,default,balance,housing,loan,contact,day,...,had_previous_contact,age_group,default_score,housing_score,loan_score,financial_burden,financial_burden_label,education_rank,education_label,job_profile
id,,,,,,,,,,,,,,,,,,,,,
1,59.0,admin.,married,secondary,no,2343,yes,no,unknown,5,...,0,Middle-Aged (51-65),0,1,0,1,Low burden,2.0,Secondary,White Collar
4,55.0,services,married,secondary,no,2476,yes,no,unknown,5,...,0,Middle-Aged (51-65),0,1,0,1,Low burden,2.0,Secondary,Blue Collar
5,54.0,admin.,married,tertiary,no,184,no,no,unknown,5,...,0,Middle-Aged (51-65),0,0,0,0,No burden,3.0,Tertiary,White Collar
6,42.0,management,single,tertiary,no,0,yes,yes,unknown,5,...,0,Adult (36-50),0,1,1,2,Medium burden,3.0,Tertiary,White Collar
8,60.0,retired,divorced,secondary,no,545,yes,no,unknown,6,...,0,Middle-Aged (51-65),0,1,0,1,Low burden,2.0,Secondary,Retired
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10638,33.0,technician,married,secondary,no,218,yes,yes,telephone,2,...,0,Young Adult (26-35),0,1,1,2,Medium burden,2.0,Secondary,White Collar
10639,42.0,management,single,tertiary,no,1146,yes,no,unknown,15,...,0,Adult (36-50),0,1,0,1,Low burden,3.0,Tertiary,White Collar
10640,31.0,unemployed,single,unknown,no,167,no,no,cellular,20,...,0,Young Adult (26-35),0,0,0,0,No burden,NaN,Unknown,Unemployed/Unknown


# CSV export

In [52]:
# Drop auto-generated index column if it was imported as a column
if "index" in df.columns:
    df = df.drop(columns=["index"])

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
 

# Drop intermediate scoring columns
cols_to_drop = [
    "default_score",
    "housing_score", 
    "loan_score"
]

df_export = df.drop(columns=cols_to_drop)

# Export transformed dataset
output_dir = Path("../Data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "bank_dataset_transformed_2026-06-29.csv"

# Export
df_export.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved to: {output_file.resolve()}")
print(f"Shape: {df_export.shape}")
print(f"Columns: {df_export.columns.tolist()}")

CSV saved to: C:\Users\nowan\Documents\itacademy\Simulador\ProjecteData\Equip_32\Data\bank_dataset_transformed_2026-06-29.csv
Shape: (9519, 25)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit', 'no_previous_contact', 'had_previous_contact', 'age_group', 'financial_burden', 'financial_burden_label', 'education_rank', 'education_label', 'job_profile']
